# Initialisation

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import lit, col, trim

#Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")

# Silver Transformation

## Trimming

In [0]:
for fields in df.schema:
  if isinstance(fields.dataType, StringType):
    df = df.withColumn(fields.name, trim(col(fields.name)))

## Customer ID Cleanup

In [0]:
df = df.withColumn("cid", F.regexp_replace(col("cid"), "-", ""))

## Country Normalization

In [0]:
df = df.withColumn(
    "cntry",
    F.when(col("cntry") == "DE", "Germany")
    .when(col("cntry").isin("US", "USA"), "United States")
    .when((col("cntry") == "") | col('cntry').isNull(), "n/a")
    .otherwise(col("cntry"))
)

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cid" : "customer_id",
    "cntry" : "country"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.display()

# Write to Silver Table

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.silver.erp_customer_location")

# Sanity Check of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_location LIMIT 10